In [2]:
import pandas as pd
import sys
import os
import scanpy as sc

In [3]:
adata = sc.read_h5ad(r"D:\Trapecar\250307_gut_liver_blood_ultimate_annotated.h5ad")

In [ ]:
adata_TCR = adata[adata.obs['chain_pairing'].isin(["single pair", "extra VJ","extra VDJ","two full chains"]),:]
adata_ab = adata_TCR[adata_TCR.obs['general type'].isin(['TCRab CD4','TCRab CD8aa','TCRab CD8ab']),:]
adata_ab.obs['clone_code'] = adata_ab.obs['TRAV'].astype(str).map(str)+' '+adata_ab.obs['TRBV'].astype(str).map(str)+' '+adata_ab.obs['cdr3a'].astype(str).map(str)+' '+ adata_ab.obs['cdr3b'].astype(str).map(str)
adata_ab.obs['subject:condition']= adata_ab.obs['Donor ID'].astype(str).map(str) + ':' + adata_ab.obs['tissue+celltype'].astype(str).map(str)

C:\Users\andre\AppData\Local\Temp\ipykernel_44920\2864242217.py:3: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata_ab.obs['clone_code'] = adata_ab.obs['TRAV'].astype(str).map(str)+' '+adata_ab.obs['TRBV'].astype(str).map(str)+' '+adata_ab.obs['cdr3a'].astype(str).map(str)+' '+ adata_ab.obs['cdr3b'].astype(str).map(str)


In [20]:
# from conga/tcrdist
clone_counts= pd.read_csv(r"G:\My Drive\result\publication\cellreport\revision\conga\gut_liver_TRM_clones_with0.tsv",sep = '\t',index_col = 0)
temp_dict_df = clone_counts[clone_counts['clone_size']>0][['clone_id','va_gene','vb_gene','cdr3a','cdr3b']]
temp_dict_df['clone_code'] = temp_dict_df['va_gene'].astype(str).map(str) + ' ' + temp_dict_df['vb_gene'].astype(str).map(str) + ' ' + temp_dict_df['cdr3a'].astype(str).map(str) + ' ' + temp_dict_df['cdr3b'].astype(str).map(str)
clone_replace_dict = temp_dict_df[['clone_id','clone_code']].set_index('clone_code').sort_index()['clone_id'].to_dict()

In [29]:
os.chdir(r"G:\My Drive\result\publication\cellreport\revision\GLIPH2")
names = ['TCRab CD4','TCRab CD8aa','TCRab CD8ab']
for i in names:
    adata_slice = adata_ab[adata_ab.obs['general type'] == i,:]

    clone_df = adata_slice.obs[['subject:condition','TRAV', 'TRBV','TRBJ', 'cdr3a', 'cdr3b','clone_code']]
    clone_counts = clone_df.groupby(['subject:condition','clone_code']).size().reset_index(name='clone frequency')
    clone_counts[['TRAV','TRBV','cdr3a','cdr3b']] = clone_counts['clone_code'].str.split(' ', expand=True)
    
    Jmap = clone_df[['TRBJ','clone_code']].drop_duplicates()
    Jmap = Jmap.set_index('clone_code')
    Jmap_bdict = Jmap['TRBJ'].to_dict()
    clone_counts['TRBJ'] =clone_counts['clone_code'].replace(Jmap_bdict)

    clone_counts = clone_counts[clone_counts['clone frequency'] >= 1]
    clone_counts['clone_id'] = clone_counts['clone_code'].map(clone_replace_dict)
    clone_counts[['cdr3b','TRBV','TRBJ','cdr3a','subject:condition','clone frequency']].to_csv(i+'_GLIPH2.tsv', sep="\t",index = False, header = False)

In [28]:
clone_counts

,subject:condition,clone_code,clone frequency,TRAV,TRBV,cdr3a,cdr3b,clone_id
0,Donor AJD3280:IEL TCRab CD8ab TRM,TRAV1-1 TRBV11-1 CAVNTNAGKSTF CASSLPNEKLFF,1,TRAV1-1,TRBV11-1,CAVNTNAGKSTF,CASSLPNEKLFF,clonotype6
1,Donor AJD3280:IEL TCRab CD8ab TRM,TRAV1-1 TRBV11-2 CAAHTNAGKSTF CASTSRDRGLHEQYF,1,TRAV1-1,TRBV11-2,CAAHTNAGKSTF,CASTSRDRGLHEQYF,clonotype7
2,Donor AJD3280:IEL TCRab CD8ab TRM,TRAV1-1 TRBV11-2 CAVETNAGKSTF CASSDRGQGGANVLTF,2,TRAV1-1,TRBV11-2,CAVETNAGKSTF,CASSDRGQGGANVLTF,clonotype9
3,Donor AJD3280:IEL TCRab CD8ab TRM,TRAV1-1 TRBV11-2 CAVSTNAGKSTF CASSVPNEKLFF,3,TRAV1-1,TRBV11-2,CAVSTNAGKSTF,CASSVPNEKLFF,clonotype11
4,Donor AJD3280:IEL TCRab CD8ab TRM,TRAV1-1 TRBV11-2 CAVYSGYSTLTF CASSLVIAGTSDTQYF,1,TRAV1-1,TRBV11-2,CAVYSGYSTLTF,CASSLVIAGTSDTQYF,clonotype12
...,...,...,...,...,...,...,...,...
6707,Donor AJKQ118:PB TCRab CD8ab Teff,TRAV8-6 TRBV28 CAVATGANNLFF CAAARGYGNQPQHF,1,TRAV8-6,TRBV28,CAVATGANNLFF,CAAARGYGNQPQHF,clonotype17694
6708,Donor AJKQ118:PB TCRab CD8ab Teff,TRAV8-6 TRBV5-1 CAVALKTSYDKVIF CASSSRVNRETQYF,1,TRAV8-6,TRBV5-1,CAVALKTSYDKVIF,CASSSRVNRETQYF,clonotype17753
6709,Donor AJKQ118:PB TCRab CD8ab Teff,TRAV8-6 TRBV5-1 CAVRISGGSYIPTF CASSFLYNSPLHF,1,TRAV8-6,TRBV5-1,CAVRISGGSYIPTF,CASSFLYNSPLHF,clonotype17763
6710,Donor AJKQ118:PB TCRab CD8ab Teff,TRAV8-6 TRBV6-6 CAVSDLYNAGNMLTF CASKGTSDTEAFF,1,TRAV8-6,TRBV6-6,CAVSDLYNAGNMLTF,CASKGTSDTEAFF,clonotype17843


In [26]:
adata_ab.obs['general type']

AAACCTGAGCGCCTCA-1-3-PB       TCRab CD4
AAACCTGAGGCAAAGA-1-3-PB     TCRab CD8ab
AAACCTGAGTTATCGC-1-3-PB       TCRab CD4
AAACCTGCAAGCGATG-1-3-PB       TCRab CD4
AAACCTGGTTACGTCA-1-3-PB       TCRab CD4
                               ...     
TTTGTCAAGAACTGTA-1-5-IEL    TCRab CD8ab
TTTGTCAAGGGAGTAA-1-5-IEL    TCRab CD8ab
TTTGTCAAGTTGTCGT-1-5-IEL      TCRab CD4
TTTGTCACAATGACCT-1-5-IEL    TCRab CD8ab
TTTGTCACATCGATTG-1-5-IEL    TCRab CD8ab
Name: general type, Length: 26297, dtype: category
Categories (3, object): ['TCRab CD4', 'TCRab CD8aa', 'TCRab CD8ab']

### Mapping GLIPH2 reulst back

In [ ]:
CD4_GLIPH2_path = pd.read_csv(r"C:\Users\andre\Documents\GitHub\gut-liver-TRM\Revision\GLIPH2\GLIPH2_results\CD4_GLIPH2.csv")

In [ ]:
names = ['TCRab CD4','TCRab CD8aa','TCRab CD8ab']
for i in names:
    adata_slice = adata_ab[adata_ab.obs['general type'] == i,:]

### GLIPH clusters shared between sites

In [ ]:
results = dict()
for i in [0,1,2]:
    for gtype in set(adatas[i].obs['general type']):
        
        adata_temp = adatas[i][np.array(np.array(adatas[i].obs['general type']== gtype) & np.array(adatas[i].obs['general subtype'] != 'MAIT')),:]
        df = adata_temp.obs[['tissue+celltype','clone']]
        clone_migrations = df.groupby(['clone', 'tissue+celltype']).size().unstack().fillna(0)
        nodes = set( adata_temp.obs['tissue+celltype'])
        migration_counts = pd.DataFrame(index=list(nodes), columns=list(nodes))
        #print(migration_counts.shape)
        for nodes1 in nodes:
            for nodes2 in nodes:
                migration_counts.loc[nodes1, nodes2] = np.sum((clone_migrations[nodes1] > 0) & (clone_migrations[nodes2] > 0))
                #so I was counting the number of clones shared among 2 organs
        matrix = migration_counts.values
        # Get the upper triangular part of the matrix without the diagonal
        # k=1 means we are not including the main diagonal
        tri_upper = np.triu_indices(matrix.shape[0], k=1)

        # Map the indices to pair names and their values
        pairs_values = [(migration_counts.columns[i], migration_counts.columns[j], matrix[i, j]) for i, j in zip(*tri_upper)]

        # Filter out pairs where the first element (before the space) is the sam
        pairs_values_intersites = [(pair1, pair2, value) for pair1, pair2, value in pairs_values if pair1.split(' ')[0] != pair2.split(' ')[0] and value >1 ]
        sorted_pairs_values_intersites = sorted(pairs_values_intersites, key=lambda x: x[2])

        pairs_values_intrasites = [(pair1, pair2, value) for pair1, pair2, value in pairs_values if pair1.split(' ')[0] == pair2.split(' ')[0] and value >1]
        sorted_pairs_values_intrasites = sorted(pairs_values_intrasites, key=lambda x: x[2])

        #shared_clones_ID = [clone_migrations.index[(clone_migrations[pair1] > 0) & (clone_migrations[pair2] > 0)].tolist() for pair1, pair2, value in sorted_pairs_values_intersites]
        cells_in_clones_shared_intersites = [clone_migrations.loc[clone_migrations.index[(clone_migrations[pair1] > 0) & (clone_migrations[pair2] > 0)].tolist(),[pair1,pair2]] for pair1, pair2, value in sorted_pairs_values_intersites]
        total_cells_in_clones_shared_intersites = [(pair1, pair2, value,np.sum(np.sum(clone_migrations.loc[clone_migrations.index[(clone_migrations[pair1] > 0) & (clone_migrations[pair2] > 0)].tolist(),[pair1,pair2]],0))) for pair1, pair2, value in sorted_pairs_values_intersites]

        #shared_clones_ID = [clone_migrations.index[(clone_migrations[pair1] > 0) & (clone_migrations[pair2] > 0)].tolist() for pair1, pair2, value in sorted_pairs_values_intersites]
        cells_in_clones_shared_intrasites = [clone_migrations.loc[clone_migrations.index[(clone_migrations[pair1] > 0) & (clone_migrations[pair2] > 0)].tolist(),[pair1,pair2]] for pair1, pair2, value in sorted_pairs_values_intrasites]
        total_cells_in_clones_shared_intrasites = [(pair1, pair2, value,np.sum(np.sum(clone_migrations.loc[clone_migrations.index[(clone_migrations[pair1] > 0) & (clone_migrations[pair2] > 0)].tolist(),[pair1,pair2]],0))) for pair1, pair2, value in sorted_pairs_values_intrasites]
        
        migration_counts[migration_counts<6] = 0
        
        results[str(i)+'_'+gtype+'_migration_counts_df'] = migration_counts
        results[str(i)+'_'+gtype+'_sorted_pairs_values_intersites'] = sorted_pairs_values_intersites
        results[str(i)+'_'+gtype+'_sorted_pairs_values_intrasites']= sorted_pairs_values_intrasites
        results[str(i)+'_'+gtype+'_cells_in_clones_shared_intersites'] =cells_in_clones_shared_intersites
        results[str(i)+'_'+gtype+'_total_cells_in_clones_shared_intersites'] =total_cells_in_clones_shared_intersites
        results[str(i)+'_'+gtype+'_cells_in_clones_shared_intrasites'] =cells_in_clones_shared_intrasites
        results[str(i)+'_'+gtype+'_total_cells_in_clones_shared_intrasites'] =total_cells_in_clones_shared_intrasites

#### Preparation for R chord diagram

In [ ]:
color_dict = dict()
for i in range(0,len(adata_all.obs['tissue+celltype'].values.categories)):
    tissue_celltype = adata_all.obs['tissue+celltype'].values.categories[i]
    celltype = np.unique(adata_all.obs['celltype'][adata_all.obs['tissue+celltype'] == tissue_celltype])[0]
    color_id = np.where(adata_all.obs['celltype'].values.categories == celltype)[0][0]
    color_dict[adata_all.obs['tissue+celltype'].values.categories[i]] = celltype_palette[color_id]

In [ ]:
for i in [0,1,2]:
    for gtype in ['TCRab CD4','TCRab CD8ab']:
        filename = 'E:/AAA_Labwork/T cells/v2/'+str(i)+'_'+gtype+'_chord.rds'
        print(filename)
        saved_result = results[str(i)+'_'+gtype+'_migration_counts_df'].values
        saved_result[saved_result<3]=0
        names =  results[str(i)+'_'+gtype+'_migration_counts_df'].columns
        %R -i saved_result
        %R -i names
        %R -i filename
        %R rownames(saved_result) = names
        %R colnames(saved_result) = names
        %R saveRDS(saved_result, file = filename)

E:/AAA_Labwork/T cells/v2/0_TCRab CD4_chord.rds
E:/AAA_Labwork/T cells/v2/0_TCRab CD8ab_chord.rds
E:/AAA_Labwork/T cells/v2/1_TCRab CD4_chord.rds
E:/AAA_Labwork/T cells/v2/1_TCRab CD8ab_chord.rds
E:/AAA_Labwork/T cells/v2/2_TCRab CD4_chord.rds
E:/AAA_Labwork/T cells/v2/2_TCRab CD8ab_chord.rds


see chord_all.R for next step

In [ ]:
all_cells = []
for i in range(0,3):
    gtype = 'TCRab CD8ab'
    adata_temp = adatas[i][adatas[i].obs['general type']== gtype,:]
    df = adata_temp.obs[['tissue+celltype','clone']]
    clone_migrations = df.groupby(['clone', 'tissue+celltype']).size().unstack().fillna(0)
    
    clone_migrations_pattern = clone_migrations>0
    clone_migrations_pattern = clone_migrations_pattern.astype(int)
    clone_migrations = clone_migrations_pattern.loc[clone_migrations_pattern.apply(sum, axis=1)>2,:]
    
    clone_migrations['pattern'] = clone_migrations.apply(tuple, axis=1)

    # Step 2: Count occurrences of each unique pattern
    pattern_counts = clone_migrations['pattern'].value_counts()

    # Step 3: Sort by frequency (this is done by value_counts automatically)
    pattern_ranking = pattern_counts.reset_index()
    pattern_ranking.columns = ['pattern', 'frequency']

    # Display the ranking
    print_full(list(clone_migrations_pattern.columns))
    print(pattern_ranking)

['IEL TCRab CD8ab TRM', 'L TCRab CD8ab MAIT', 'L TCRab CD8ab Naive/TCM', 'L TCRab CD8ab TRM', 'L TCRab CD8ab Teff', 'LP TCRab CD8ab TCM', 'LP TCRab CD8ab TEM', 'LP TCRab CD8ab TRM', 'PB TCRab CD8ab Naive/TCM', 'PB TCRab CD8ab TEM', 'PB TCRab CD8ab Teff']
                              pattern  frequency
0   (0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1)          9
1   (1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0)          3
2   (1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1)          3
3   (1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0)          2
4   (0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 1)          1
5   (1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1)          1
6   (0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1)          1
7   (1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0)          1
8   (0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0)          1
9   (1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0)          1
10  (0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1)          1
11  (0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1)          1
12  (0, 0, 1, 1, 1, 0, 1, 0, 0, 0, 0)          1
13  (1, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0)          1
14  (1, 0,